# Navier–Stokes PINNsFormer + Adam simple-sum baseline

这是 `navier_stokes_config.ipynb` 的**公平对照组**：seed、800 个训练点、5-step pseudo-time、网络结构、PDE、黏性系数、1000 optimizer steps 与 `Adam(lr=1e-4)` 全部保持一致，唯一差别是这里直接对 `L_data + L_physics` 反向传播，不使用 ConFIG。


In [ ]:
from pathlib import Path
import os, sys, time, random
import numpy as np
import torch
import matplotlib.pyplot as plt
from tqdm import tqdm

PINNSFORMER_ROOT = Path(os.environ.get("PINNSFORMER_ROOT", "/home/simplexity/cyt/pinnsformer-main"))
DATA_PATH = PINNSFORMER_ROOT / "demo" / "navier_stokes" / "cylinder_nektar_wake.mat"

from gradient_diagnostics import gradient_vector, ConflictTracker
from navier_stokes_common import PINNsformer, init_weights, get_n_params, load_training_data, compute_ns_losses, evaluation_tensors

assert DATA_PATH.exists(), DATA_PATH


In [ ]:
SEED=0
DEVICE="cuda:0"
N_TRAIN=800
NUM_STEP=5
TIME_STEP=1e-2
EPOCHS=1000
LR=1e-4
DIAG_INTERVAL=10

np.random.seed(SEED); random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed(SEED)
assert torch.cuda.is_available()
device=torch.device(DEVICE)
batch, reference = load_training_data(DATA_PATH, device, seed=SEED, n_train=N_TRAIN, num_step=NUM_STEP, time_step=TIME_STEP)
print(torch.cuda.get_device_name(device), batch["x"].shape)


In [ ]:
model=PINNsformer(d_out=2,d_hidden=512,d_model=32,N=1,heads=2).to(device)
model.apply(init_weights)
optimizer=torch.optim.Adam(model.parameters(),lr=LR)
print("params:",get_n_params(model))


In [ ]:
torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats(device)
history=[]
tracker=ConflictTracker("data","physics")
start=time.time()

for step in tqdm(range(EPOCHS)):
    losses=compute_ns_losses(model,batch["x"],batch["y"],batch["t"],batch["u"],batch["v"])

    # Diagnostics only every DIAG_INTERVAL steps so the baseline is not forced to
    # pay two extra loss-specific gradient calculations at every update.
    cosine=float("nan")
    g_data_norm=float("nan")
    g_physics_norm=float("nan")
    if step % DIAG_INTERVAL == 0 or step == EPOCHS-1:
        g_data=gradient_vector(losses["data"],model,retain_graph=True)
        g_physics=gradient_vector(losses["physics"],model,retain_graph=True)
        cosine=tracker.update({"data":g_data,"physics":g_physics})
        g_data_norm=float(g_data.norm())
        g_physics_norm=float(g_physics.norm())

    optimizer.zero_grad(set_to_none=True)
    losses["total"].backward()
    optimizer.step()

    if step % DIAG_INTERVAL == 0 or step == EPOCHS-1:
        history.append({"step":step,"total":float(losses["total"].detach()),"data":float(losses["data"].detach()),"physics":float(losses["physics"].detach()),"g_data_norm":g_data_norm,"g_physics_norm":g_physics_norm,"cosine":cosine,"conflict":int(np.isfinite(cosine) and cosine<0)})

torch.cuda.synchronize()
elapsed=time.time()-start
print(f"wall time: {elapsed:.2f}s ({elapsed/60:.2f}min)")
print(f"peak allocated: {torch.cuda.max_memory_allocated(device)/1024**2:.2f} MiB")
print("diagnostic conflict:",tracker.summary())


In [ ]:
OUTPUT_DIR=Path("./outputs/navier_stokes_adam_baseline"); OUTPUT_DIR.mkdir(parents=True,exist_ok=True)
MODEL_PATH=OUTPUT_DIR/"ns_pinnsformer_adam_sum.pt"
HISTORY_PATH=OUTPUT_DIR/"ns_pinnsformer_adam_sum_history.npz"
torch.save(model.state_dict(),MODEL_PATH)
keys=list(history[0]); np.savez(HISTORY_PATH,**{k:np.asarray([r[k] for r in history]) for k in keys})
print(MODEL_PATH.resolve()); print(HISTORY_PATH.resolve())


## Evaluation

为与新 ConFIG notebook 一致，这里也按训练定义 `u=psi_y, v=-psi_x` 评估。

In [ ]:
ev, truth = evaluation_tensors(reference,device,snap=100,num_step=NUM_STEP,time_step=TIME_STEP)
psi_p=model(ev["x"],ev["y"],ev["t"]); psi=psi_p[:,:,0:1]; p_pred=psi_p[:,:,1:2]
u_pred=torch.autograd.grad(psi,ev["y"],torch.ones_like(psi),retain_graph=True)[0]
v_pred=-torch.autograd.grad(psi,ev["x"],torch.ones_like(psi))[0]
u_pred=u_pred.detach().cpu().numpy()[:,0]; v_pred=v_pred.detach().cpu().numpy()[:,0]; p_pred=p_pred.detach().cpu().numpy()[:,0]
for name,pred in [("u",u_pred),("v",v_pred),("p",p_pred)]:
    err=np.linalg.norm(truth[name]-pred,2)/np.linalg.norm(truth[name],2)
    print(f"{name} relative L2 = {err:.6f}")
